<a href="https://colab.research.google.com/github/MonikaBarget/DigitalHistory/blob/master/StoryScript_buildConversation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Conversation builder for StoryScript**

With the code below, you can build a conversation with a person using input boxes and buttons. This means that you do not have to write the person.html file and the person.ts file manually. The script will generate both based on the information you provide and put them into a .zip archive that you can download.

The easiest way to run the script is to use the ```Run all``` button above. You can also execute the cells below individually. The installation is set to  quiet, meaning that you will see no output until the import of all packages is complete.

In [ ]:
# Install required packages
!pip install -q ipywidgets

import os
import zipfile
import json
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from google.colab import files

print("Installation complete!")

After the successful installation, we now define *global variables*, meaning a placeholder that is visible throughout the program. You will fill the variables with your own input every time you run the script!

In [ ]:
# Global variables to store user inputs
person_name = ""
qa_pairs = []
ts_params = {}
person_description = ""

print("Global variables set!")

Now we run the main part of the script, which is divided into several functions. A function is a section of code that can return data as a result and can easily be re-run to avoid code repetitions. You will see options to input your data. Use the grey buttons below the code to interact with the script!

In [ ]:
# Step 1: Collect Person Information
def step1():
    global person_name, person_description
    name_input = widgets.Text(
        description="Person Name:",
        placeholder="Enter the person's name (e.g., Moser)",
        layout=widgets.Layout(width='50%')
    )
    description_input = widgets.Textarea(
        description="Description:",
        placeholder="Enter the person's description (HTML allowed)",
        layout=widgets.Layout(width='80%', height='100px')
    )

    def on_submit(b):
        global person_name, person_description
        person_name = name_input.value.strip()
        person_description = description_input.value.strip()
        if person_name:
            display(HTML(f"<h4>Person Name: {person_name}</h4>"))
            step2()

    submit_button = widgets.Button(description="Next")
    submit_button.on_click(on_submit)

    display(HTML("<h3>Step 1: Enter Person Details</h3>"))
    display(name_input, description_input, submit_button)

# Step 2: Collect Q&A Pairs
def step2():
    qa_pairs.clear()
    question_input = widgets.Text(
        description="Question:",
        placeholder="Enter a question for the person!",
        layout=widgets.Layout(width='80%')
    )
    answer_input = widgets.Textarea(
        description="Answer:",
        placeholder="Enter the person's answer (HTML allowed)!",
        layout=widgets.Layout(width='80%', height='100px')
    )
    qa_list = widgets.HTML(value="<h5>No Q&A pairs added yet</h5>")

    def add_qa(b):
        question = question_input.value.strip()
        answer = answer_input.value.strip()
        if question and answer:
            qa_pairs.append((question, answer))
            question_input.value = ""
            answer_input.value = ""
            update_qa_list()

    def update_qa_list():
        if qa_pairs:
            html = "<h5>Added Q&A Pairs:</h5><ol>"
            for q, a in qa_pairs:
                html += f"<li><strong>Q:</strong> {q}<br><strong>A:</strong> {a[:50]}...</li>"
            html += "</ol>"
            qa_list.value = html
        else:
            qa_list.value = "<h5>No Q&A pairs added yet</h5>"

    def on_done(b):
        if not qa_pairs:
            print("Please add at least one Q&A pair and confirm with the button!")
            return
        display(HTML(f"<h4>Added {len(qa_pairs)} Q&A pairs</h4>"))
        step3()

    add_button = widgets.Button(description="Add Q&A Pair")
    done_button = widgets.Button(description="Done with Q&A")

    add_button.on_click(add_qa)
    done_button.on_click(on_done)

    display(HTML("<h3>Step 2: Add Q&A Pairs</h3>"))
    display(question_input, answer_input, add_button, done_button, qa_list)

# Step 3: Collect TS File Parameters
def step3():
    global ts_params

    hitpoints_input = widgets.IntText(
        description="Hitpoints:",
        value=1,
        min=0,
        layout=widgets.Layout(width='30%')
    )
    attack_input = widgets.Text(
        description="Attack:",
        value='0',
        layout=widgets.Layout(width='30%')
    )
    can_attack_input = widgets.Dropdown(
        description="Can Attack:",
        options=[('No', False), ('Yes', True)],
        value=False,
        layout=widgets.Layout(width='30%')
    )
    currency_input = widgets.IntText(
        description="Currency:",
        value=0,
        min=0,
        layout=widgets.Layout(width='30%')
    )
    items_input = widgets.Text(
        description="Items (comma-separated):",
        placeholder="e.g., Sword(), Key()",
        layout=widgets.Layout(width='80%')
    )
    special_actions_input = widgets.Textarea(
        description="Special Actions (JSON format):",
        placeholder='e.g., [["addItem", (game, person) => { game.addItem("key"); }]]',
        layout=widgets.Layout(width='80%', height='100px')
    )

    def on_submit(b):
        global ts_params # Ensure the global ts_params is updated
        ts_params = {
            "hitpoints": hitpoints_input.value,
            "attack": attack_input.value,
            "can_attack": can_attack_input.value,
            "currency": currency_input.value,
            "items": items_input.value,
            "special_actions": special_actions_input.value
        }
        generate_files()

    submit_button = widgets.Button(description="Generate Files")
    submit_button.on_click(on_submit)

    display(HTML("<h3>Step 3: Enter TS File Parameters</h3>"))
    display(hitpoints_input, attack_input, can_attack_input, currency_input, items_input, special_actions_input, submit_button)

# Generate the files
def generate_files():
    # Create HTML File
    html_content = f"""<img class=\"picture\" src=\"resources/{person_name.lower()}.png\" />
<description>
    <p>
        {person_description}
    </p>
</description>
<conversation>
    <default-reply>
        I have nothing more to say. Farewell.
    </default-reply>
    <node name=\"start\">
        <p>
            {qa_pairs[0][1] if qa_pairs else "Hello. How can I help you?"}
        </p>
        <replies>
"""
    for i, (question, _) in enumerate(qa_pairs[1:], start=1):
        html_content += f"""            <reply node=\"node{i}\">
                {question}
            </reply>
"""
    html_content += """        </replies>
    </node>
"""
    for i, (_, answer) in enumerate(qa_pairs[1:], start=1):
        html_content += f"""    <node name=\"node{i}\">
        <p>
            {answer}
        </p>
        <replies>
            <reply node=\"start\">
                Back to start.
            </reply>
        </replies>
    </node>
"""
    html_content += """</conversation>
"""

    with open(f"{person_name}.html", "w") as f:
        f.write(html_content)

    # Create TS File
    # Use .get() to safely access values with default empty strings if keys are missing
    items_value = ts_params.get('items', '')
    items_list = f"[{items_value}]" if items_value else "[]"

    special_actions_value = ts_params.get('special_actions', '')
    actions_list = special_actions_value if special_actions_value else "[]"

    ts_content = f"""// src/Games/MyInteractiveMap/persons/{person_name}.ts
import {{ IPerson, Person }} from '../types';
import conversation from './{person_name}.html?raw';

export function {person_name}(): IPerson {{
    return Person({{
        name: '{person_name}',
        description: conversation,
        hitpoints: {ts_params.get('hitpoints', 1)}, # Default hitpoints to 1
        attack: '{ts_params.get('attack', '0')}', # Default attack to '0'
        canAttack: {ts_params.get('can_attack', False)}, # Default canAttack to False
        items: {items_list},
        currency: {ts_params.get('currency', 0)}, # Default currency to 0
        conversation: {{
            actions: {actions_list}
        }},
        quests: []
    }});
}}
"""

    with open(f"{person_name}.ts", "w") as f:
        f.write(ts_content)

    # Create ZIP file
    with zipfile.ZipFile(f"{person_name}_files.zip", 'w') as zipf:
        zipf.write(f"{person_name}.html")
        zipf.write(f"{person_name}.ts")

    # Display download button and instructions
    display(HTML("<h3>Files Generated Successfully!</h3>"))
    files.download(f"{person_name}_files.zip")

    instructions = f"""
    <h3>Instructions:</h3>
    <ol>
        <li>Download the ZIP file above.</li>
        <li>Extract the files to your project:</li>
        <ul>
            <li>Copy <code>{person_name}.html</code> to <code>src/Games/MyInteractiveMap/persons/</code></li>
            <li>Copy <code>{person_name}.ts</code> to <code>src/Games/MyInteractiveMap/persons/</code></li>
        </ul>
        <li>Add the person to a location file (e.g., <code>Swabia.ts</code>) under the <code>persons</code> array:</li>
        <pre>
        persons: [
            {person_name}() // Add {person_name} as a person in this location
        ]
        </pre>
        <li>Ensure you have an image file named <code>{person_name.lower()}.png</code> in your <code>resources</code> folder.</li>
    </ol>
    """
    display(HTML(instructions))

# Start the process
step1()